# c2-activity · Modelo, consulta y limpieza de datos
**Cristhian Gaitán Guzmán** · Ciencia de Datos (Cód. 69109) · CORHUILA 2026-B · Semana 9 (Corte 2)

Cuaderno **ejecutado** con las tres partes de la actividad: (1) base de datos que implementa el ERD, (2) limpieza con pandas con reporte antes/después y (3) dos preguntas con consultas SQL y pandas. El ERD (Mermaid) y la sección *Data & cleaning* en inglés están en el `README.md`.

**Caso:** planta de empaques del Huila (4 máquinas, 2 líneas, 3 turnos). Pregunta de negocio: *¿qué máquina y qué turno concentran los defectos, y a cuál le hacemos mantenimiento primero?* Dataset ilustrativo del taller de la Semana 9 [2], no datos reales de una empresa.

> Para ejecutarlo, abre el cuaderno desde la carpeta `c2-activity/` (necesita `data/registro_produccion.csv` y `schema.sql`).

In [1]:
import re, sqlite3
from pathlib import Path
import pandas as pd

BASE = Path.cwd()                      # carpeta c2-activity/
DATA = BASE / "data"
MAQUINAS_VALIDAS = ["M-01", "M-02", "M-03", "M-04"]
TURNOS_VALIDOS = ["Mañana", "Tarde", "Noche"]
CAPACIDAD_MAX = 1800                   # unidades por turno
SENSOR_MIN, SENSOR_MAX = 15, 60        # °C
print("pandas", pd.__version__)

pandas 3.0.2


## 1. Carga y diagnóstico inicial (antes de tocar nada)
El archivo original **no se modifica**: toda la limpieza se hace sobre una copia.

In [2]:
df = pd.read_csv(DATA / "registro_produccion.csv")
print("Filas y columnas:", df.shape)
display(df.sample(8, random_state=7))
print(df.dtypes.to_string())
print("\nNulos por columna:")
print(df.isna().sum().to_string())
print("\nFilas duplicadas:", df.duplicated().sum())
print("Valores distintos en 'maquina':", df["maquina"].nunique(), "(esperado 4)")
print("Valores distintos en 'turno'  :", df["turno"].nunique(), "(esperado 3)")
con_barra = df["fecha"].str.contains("/")
temp_num = pd.to_numeric(df["temperatura"], errors="coerce")
coma_decimal = temp_num.isna() & df["temperatura"].notna()
print("Fechas dd/mm/aaaa:", con_barra.sum(), "| temperaturas con coma decimal:", coma_decimal.sum(),
      "| lecturas > 60 (°F):", (temp_num > 60).sum())

Filas y columnas: (297, 6)


,fecha,turno,maquina,producidas,defectuosas,temperatura
199,2026-09-25,MAÑANA,M-04,1276,14.0,30.7
3,2026-09-07,Mañana,M-04,1054,18.0,31.2
216,26/09/2026,Tarde,M-04,1034,17.0,31.3
284,03/10/2026,Mañana,M-02,1107,21.0,30.7
125,2026-09-18,Mañana,M-04,1126,11.0,30.5
243,2026-09-29,Noche,M-01,18,1295.0,32.2
78,2026-09-14,Tarde,M-01,1069,8.0,34.2
99,2026-09-16,Mañana,M-02,1177,27.0,88.0


fecha              str
turno              str
maquina            str
producidas       int64
defectuosas    float64
temperatura        str

Nulos por columna:
fecha           0
turno           0
maquina         0
producidas      0
defectuosas    10
temperatura     6

Filas duplicadas: 9
Valores distintos en 'maquina': 16 (esperado 4)
Valores distintos en 'turno'  : 12 (esperado 3)
Fechas dd/mm/aaaa: 30 | temperaturas con coma decimal: 20 | lecturas > 60 (°F): 14


### Funciones de medición
Se usan **antes y después**, de modo que la comparación mide exactamente lo mismo. Las cinco dimensiones: completitud, unicidad, consistencia, formato y validez [6].

In [3]:
def reporte_calidad(tabla):
    """% de filas que cumple cada dimensión de calidad."""
    prod = pd.to_numeric(tabla["producidas"], errors="coerce")
    defe = pd.to_numeric(tabla["defectuosas"], errors="coerce")
    temp = pd.to_numeric(tabla["temperatura"], errors="coerce")
    cumple_reglas = (prod.between(1, CAPACIDAD_MAX) & defe.between(0, prod)
                     & temp.between(SENSOR_MIN, SENSOR_MAX))
    medidas = {
        "Completitud": tabla.notna().all(axis=1).mean(),
        "Unicidad": (~tabla.duplicated()).mean(),
        "Consistencia": (tabla["maquina"].isin(MAQUINAS_VALIDAS) & tabla["turno"].isin(TURNOS_VALIDOS)).mean(),
        "Formato": tabla["fecha"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").mean(),
        "Validez": cumple_reglas.mean(),
    }
    return (pd.Series(medidas) * 100).round(1)


def perfil(tabla):
    temp_texto = pd.api.types.is_string_dtype(tabla["temperatura"]) or tabla["temperatura"].dtype == object
    return {
        "Filas": len(tabla),
        "Celdas vacías (nulos)": int(tabla.isna().sum().sum()),
        "Filas con algún vacío": int(tabla.isna().any(axis=1).sum()),
        "Filas duplicadas": int(tabla.duplicated().sum()),
        "Valores distintos en 'maquina' (esperado 4)": int(tabla["maquina"].nunique()),
        "Valores distintos en 'turno' (esperado 3)": int(tabla["turno"].nunique()),
        "Fechas fuera de aaaa-mm-dd": int((~tabla["fecha"].astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}")).sum()),
        "Tipo de 'temperatura'": "texto" if temp_texto else "numérico",
        "Tipo de 'fecha'": "fecha" if pd.api.types.is_datetime64_any_dtype(tabla["fecha"]) else "texto",
    }


bitacora = []
def registrar(paso, accion, antes, despues, corregidos=0):
    bitacora.append({"paso": paso, "accion": accion, "filas_antes": antes, "filas_despues": despues,
                     "filas_eliminadas": antes - despues, "valores_corregidos": corregidos})
    print(f"[{paso}] {accion:<52} filas {antes} -> {despues} | corregidos: {corregidos}")


calidad_antes, perfil_antes = reporte_calidad(df), perfil(df)
calidad_antes

Completitud     94.6
Unicidad        97.0
Consistencia    88.9
Formato         89.9
Validez         80.8
dtype: float64

## 2. Limpieza paso a paso (siempre sobre una copia)
**Paso 1 · Duplicados** (unicidad) y **Paso 2 · textos inconsistentes** (consistencia): se quitan las copias exactas y se homologan los códigos de máquina y turno con un diccionario.

In [4]:
limpio = df.copy()

# Paso 1 · duplicados
antes = len(limpio)
limpio = limpio.drop_duplicates().reset_index(drop=True)
registrar("Paso 1", "Quitar filas duplicadas", antes, len(limpio))

# Paso 2 · homologar textos
maq_orig, tur_orig = limpio["maquina"].copy(), limpio["turno"].copy()
limpio["maquina"] = limpio["maquina"].str.strip().str.upper()
limpio["maquina"] = limpio["maquina"].replace({
    "M01": "M-01", "M02": "M-02", "M03": "M-03", "M04": "M-04",
    "M-1": "M-01", "M-2": "M-02", "M-3": "M-03", "M-4": "M-04"})
limpio["turno"] = limpio["turno"].str.strip().str.capitalize().replace({"Manana": "Mañana"})
cambios_texto = int((limpio["maquina"] != maq_orig).sum() + (limpio["turno"] != tur_orig).sum())
registrar("Paso 2", "Homologar máquinas y turnos", len(limpio), len(limpio), cambios_texto)
assert limpio.duplicated().sum() == 0, "Aparecieron duplicados nuevos al homologar"
print("Máquinas:", sorted(limpio["maquina"].unique()), "| Turnos:", sorted(limpio["turno"].unique()))

[Paso 1] Quitar filas duplicadas                              filas 297 -> 288 | corregidos: 0
[Paso 2] Homologar máquinas y turnos                          filas 288 -> 288 | corregidos: 33
Máquinas: ['M-01', 'M-02', 'M-03', 'M-04'] | Turnos: ['Mañana', 'Noche', 'Tarde']


**Paso 3 · Tipos y formatos.** La temperatura venía como texto por la coma decimal; las fechas mezclaban `aaaa-mm-dd` y `dd/mm/aaaa`. Se convierte formato por formato (adivinar `07/09` es peligroso: ¿7 de septiembre o 9 de julio?) y se verifica que la conversión no vacíe datos.

**Paso 4 · Unidades.** Ninguna máquina trabaja a 95 °C: todo valor sobre 60 °C está en Fahrenheit.

In [5]:
# Paso 3 · tipos
vacios_antes = int(limpio["temperatura"].isna().sum())
limpio["temperatura"] = pd.to_numeric(
    limpio["temperatura"].astype(str).str.replace(",", ".", regex=False), errors="coerce")
assert int(limpio["temperatura"].isna().sum()) == vacios_antes, "to_numeric vació datos válidos"

iso = pd.to_datetime(limpio["fecha"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(limpio["fecha"], format="%d/%m/%Y", errors="coerce")
limpio["fecha"] = iso.fillna(dmy)
assert limpio["fecha"].isna().sum() == 0, "Quedaron fechas sin convertir"
registrar("Paso 3", "Temperatura a número y fechas a un solo formato",
          len(limpio), len(limpio), int(coma_decimal.sum() + con_barra.sum()))
print("Rango de fechas:", limpio["fecha"].min().date(), "a", limpio["fecha"].max().date())

# Paso 4 · °F -> °C
en_f = limpio["temperatura"] > SENSOR_MAX
limpio.loc[en_f, "temperatura"] = ((limpio.loc[en_f, "temperatura"] - 32) * 5 / 9).round(1)
registrar("Paso 4", "Convertir lecturas en °F a °C", len(limpio), len(limpio), int(en_f.sum()))

[Paso 3] Temperatura a número y fechas a un solo formato      filas 288 -> 288 | corregidos: 50
Rango de fechas: 2026-09-07 a 2026-10-03
[Paso 4] Convertir lecturas en °F a °C                        filas 288 -> 288 | corregidos: 14


**Paso 5 · Faltantes (completitud).** Dos decisiones distintas:
- `defectuosas` vacía → **se elimina la fila**: es lo que se quiere medir y rellenarla sería inventar defectos.
- `temperatura` vacía → **se imputa con la mediana de su máquina** y se marca en `temp_imputada` (variable de apoyo; lo estimado no se confunde con lo medido).

**Paso 6 · Valores imposibles (validez).** Se eliminan, no se "arreglan": adivinar la corrección también es inventar.

In [6]:
# Paso 5a · eliminar sin 'defectuosas'
antes = len(limpio)
limpio = limpio.dropna(subset=["defectuosas"]).reset_index(drop=True)
registrar("Paso 5a", "Eliminar filas sin dato de defectuosas", antes, len(limpio))

# Paso 5b · imputar temperatura
limpio["temp_imputada"] = limpio["temperatura"].isna()
mediana_maq = limpio.groupby("maquina")["temperatura"].transform("median")
limpio["temperatura"] = limpio["temperatura"].fillna(mediana_maq)
registrar("Paso 5b", "Imputar temperatura con la mediana de su máquina",
          len(limpio), len(limpio), int(limpio["temp_imputada"].sum()))

# Paso 6 · imposibles según las reglas del negocio
sobre_capacidad = limpio["producidas"] > CAPACIDAD_MAX
invertidas = limpio["defectuosas"] > limpio["producidas"]
negativas = limpio["defectuosas"] < 0
imposible = sobre_capacidad | invertidas | negativas
print(f"Imposibles -> sobre capacidad: {sobre_capacidad.sum()}, invertidas: {invertidas.sum()}, negativas: {negativas.sum()}")
antes = len(limpio)
limpio = limpio[~imposible].reset_index(drop=True)
limpio["producidas"] = limpio["producidas"].astype(int)
limpio["defectuosas"] = limpio["defectuosas"].astype(int)
registrar("Paso 6", "Eliminar filas que violan las reglas del negocio", antes, len(limpio))

[Paso 5a] Eliminar filas sin dato de defectuosas               filas 288 -> 278 | corregidos: 0
[Paso 5b] Imputar temperatura con la mediana de su máquina     filas 278 -> 278 | corregidos: 6
Imposibles -> sobre capacidad: 2, invertidas: 3, negativas: 2
[Paso 6] Eliminar filas que violan las reglas del negocio     filas 278 -> 271 | corregidos: 0


### Atípicos de temperatura: se investigan, no se borran
Con la regla del IQR se marcan los valores raros; luego se comprueba si son errores (fuera del rango del sensor) o lecturas reales.

In [7]:
q1, q3 = limpio["temperatura"].quantile([0.25, 0.75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atipicos = limpio[(limpio["temperatura"] < lim_inf) | (limpio["temperatura"] > lim_sup)]
print(f"Q1={q1:.1f} Q3={q3:.1f} IQR={iqr:.1f} -> límites [{lim_inf:.1f}, {lim_sup:.1f}] °C")
print("Atípicos:", len(atipicos), "| fuera del rango del sensor:", int((~atipicos['temperatura'].between(SENSOR_MIN, SENSOR_MAX)).sum()))
print("Por máquina:", atipicos["maquina"].value_counts().to_dict())
print("Por turno  :", atipicos["turno"].value_counts().to_dict())

Q1=30.7 Q3=33.7 IQR=3.0 -> límites [26.2, 38.2] °C
Atípicos: 13 | fuera del rango del sensor: 0
Por máquina: {'M-03': 13}
Por turno  : {'Tarde': 12, 'Mañana': 1}


## 3. Reporte antes / después

In [8]:
calidad_despues, perfil_despues = reporte_calidad(limpio), perfil(limpio)
comparacion = pd.concat([
    pd.DataFrame({"indicador": list(perfil_antes), "antes": list(perfil_antes.values()), "despues": list(perfil_despues.values())}),
    pd.DataFrame({"indicador": [f"Calidad · {d} (%)" for d in calidad_antes.index],
                  "antes": calidad_antes.values, "despues": calidad_despues.values}),
], ignore_index=True)
display(comparacion)
display(pd.DataFrame(bitacora))
print("Filas descartadas:", len(df) - len(limpio), f"({(len(df) - len(limpio)) / len(df):.1%})")

,indicador,antes,despues
0,Filas,297,271
1,Celdas vacías (nulos),16,0
2,Filas con algún vacío,16,0
3,Filas duplicadas,9,0
4,Valores distintos en 'maquina' (esperado 4),16,4
5,Valores distintos en 'turno' (esperado 3),12,3
6,Fechas fuera de aaaa-mm-dd,30,0
7,Tipo de 'temperatura',texto,numérico
8,Tipo de 'fecha',texto,fecha
9,Calidad · Completitud (%),94.6,100.0


,paso,accion,filas_antes,filas_despues,filas_eliminadas,valores_corregidos
0,Paso 1,Quitar filas duplicadas,297,288,9,0
1,Paso 2,Homologar máquinas y turnos,288,288,0,33
2,Paso 3,Temperatura a número y fechas a un solo formato,288,288,0,50
3,Paso 4,Convertir lecturas en °F a °C,288,288,0,14
4,Paso 5a,Eliminar filas sin dato de defectuosas,288,278,10,0
5,Paso 5b,Imputar temperatura con la mediana de su máquina,278,278,0,6
6,Paso 6,Eliminar filas que violan las reglas del negocio,278,271,7,0


Filas descartadas: 26 (8.8%)


In [9]:
limpio_out = limpio.copy()
limpio_out["fecha"] = limpio_out["fecha"].dt.strftime("%Y-%m-%d")
limpio_out.to_csv(DATA / "registro_produccion_limpio.csv", index=False)
pd.DataFrame(bitacora).to_csv(DATA / "bitacora_limpieza.csv", index=False)
comparacion.to_csv(DATA / "antes_despues.csv", index=False)
print("Guardado:", limpio_out.shape, "en data/registro_produccion_limpio.csv")

Guardado: (271, 7) en data/registro_produccion_limpio.csv


**Dos dimensiones que mejoraron de forma clara**
1. **Validez (80,8 % → 100 %)**: era la peor. Convertir 14 lecturas de °F a °C, corregir las comas decimales y descartar 7 filas imposibles hizo que todos los valores fueran posibles.
2. **Consistencia (88,9 % → 100 %)**: había 16 "máquinas" y 12 "turnos" por mayúsculas, espacios y variantes como `M3`. Tras homologar quedan 4 y 3, y por fin se puede agrupar por máquina y turno.

## 4. Base de datos del ERD (SQLite)
Se crea la base con `schema.sql` (con reglas `CHECK` y `UNIQUE`) y se carga la tabla limpia. La base rechaza por sí sola los errores que arriba se limpiaron a mano.

In [10]:
DB = DATA / "planta.db"
if DB.exists():
    DB.unlink()
con = sqlite3.connect(DB)
con.executescript((BASE / "schema.sql").read_text(encoding="utf-8"))
con.executemany("INSERT INTO linea VALUES (?,?,?)", [(1, "Línea 1", "Película"), (2, "Línea 2", "Bolsas")])
con.executemany("INSERT INTO maquina VALUES (?,?,?,?)", [
    ("M-01", "Extrusora A", 1, 2019), ("M-02", "Extrusora B", 1, 2021),
    ("M-03", "Selladora C", 2, 2012), ("M-04", "Cortadora D", 2, 2023)])
con.executemany("INSERT INTO turno VALUES (?,?)", [(1, "Mañana"), (2, "Tarde"), (3, "Noche")])

id_turno = {"Mañana": 1, "Tarde": 2, "Noche": 3}
hechos = pd.DataFrame({
    "fecha": limpio["fecha"].dt.strftime("%Y-%m-%d"),
    "id_maquina": limpio["maquina"],
    "id_turno": limpio["turno"].map(id_turno),
    "producidas": limpio["producidas"], "defectuosas": limpio["defectuosas"],
    "temperatura_c": limpio["temperatura"], "temp_imputada": limpio["temp_imputada"].astype(int)})
hechos.to_sql("registro_produccion", con, if_exists="append", index=False)
con.commit()
print("Filas cargadas:", con.execute("SELECT COUNT(*) FROM registro_produccion").fetchone()[0])
print("Integridad referencial:", con.execute("PRAGMA foreign_key_check").fetchall() or "OK")

Filas cargadas: 271
Integridad referencial: OK


## 5. Pregunta 1 — ¿Qué máquinas superan la tasa de defectos de toda la planta?
*Filtro + agregación:* `GROUP BY` por máquina y `HAVING` contra la tasa global. La tasa se calcula con **totales** (no promediando tasas por fila: un turno grande pesa más que uno pequeño).

In [11]:
texto = (BASE / "consultas.sql").read_text(encoding="utf-8")
bloques = re.split(r"^-- \[(\w+)\]\s*$", texto, flags=re.M)[1:]
consultas = dict(zip(bloques[0::2], (b.strip() for b in bloques[1::2])))

p1_sql = pd.read_sql_query(consultas["P1"], con)
display(p1_sql)
print("Contexto, ranking completo de las 4 máquinas:")
display(pd.read_sql_query(consultas["P1_CONTEXTO"], con))

,id_maquina,nombre,producidas,defectuosas,tasa_pct,veces_la_planta
0,M-03,Selladora C,81008,3475,4.29,1.94


Contexto, ranking completo de las 4 máquinas:


,id_maquina,producidas,defectuosas,tasa_pct
0,M-03,81008,3475,4.29
1,M-01,80475,1389,1.73
2,M-02,81031,1302,1.61
3,M-04,80958,982,1.21


In [12]:
# Equivalente en pandas
d = hechos.merge(pd.DataFrame({"id_turno": [1, 2, 3], "turno": ["Mañana", "Tarde", "Noche"]}), on="id_turno")
tasa_planta = d["defectuosas"].sum() / d["producidas"].sum() * 100
por_maq = d.groupby("id_maquina").agg(producidas=("producidas", "sum"), defectuosas=("defectuosas", "sum"))
por_maq["tasa_pct"] = (por_maq["defectuosas"] / por_maq["producidas"] * 100).round(2)
p1_pd = por_maq[por_maq["defectuosas"] / por_maq["producidas"] * 100 > tasa_planta].sort_values("tasa_pct", ascending=False)
print(f"Tasa de toda la planta: {tasa_planta:.2f} %")
display(p1_pd)
assert list(p1_sql["id_maquina"]) == list(p1_pd.index) and list(p1_sql["tasa_pct"]) == list(p1_pd["tasa_pct"])
print("SQL y pandas coinciden.")

Tasa de toda la planta: 2.21 %


,producidas,defectuosas,tasa_pct
id_maquina,,,
M-03,81008,3475,4.29


SQL y pandas coinciden.


**Hallazgo.** Solo la **M-03** supera la tasa de la planta (4,29 % frente a 2,21 %, casi el doble). Las otras tres están entre 1,21 % y 1,73 %. Este resultado solo es confiable tras la limpieza: en el registro sin limpiar había 16 "máquinas" y, entre los códigos bien escritos, la M-01 aparecía con 5,93 % frente a 3,58 % de la M-03, lo que habría mandado a mantenimiento la máquina equivocada.

## 6. Pregunta 2 — En la M-03, ¿en qué turno se concentran los defectos y qué temperatura traía?
*Filtro + agregación:* `WHERE id_maquina = 'M-03'` y `GROUP BY` turno. La temperatura media usa solo lecturas **medidas** (se excluyen las imputadas).

In [13]:
p2_sql = pd.read_sql_query(consultas["P2"], con)
display(p2_sql)

,turno,registros,temp_media_c,producidas,defectuosas,tasa_pct
0,Tarde,23,38.5,26923,1736,6.45
1,Mañana,22,34.5,26683,925,3.47
2,Noche,22,30.7,27402,814,2.97


In [14]:
# Equivalente en pandas + apoyo para el hallazgo
m03 = d[d["id_maquina"] == "M-03"]
p2_pd = m03.groupby("turno").agg(
    registros=("producidas", "size"),
    temp_media_c=("temperatura_c", lambda s: round(s[m03.loc[s.index, "temp_imputada"] == 0].mean(), 1)),
    producidas=("producidas", "sum"), defectuosas=("defectuosas", "sum"))
p2_pd["tasa_pct"] = (p2_pd["defectuosas"] / p2_pd["producidas"] * 100).round(2)
p2_pd = p2_pd.sort_values("tasa_pct", ascending=False)
display(p2_pd)
assert list(p2_sql["turno"]) == list(p2_pd.index) and list(p2_sql["tasa_pct"]) == list(p2_pd["tasa_pct"])
print("SQL y pandas coinciden.\n")

medidas = d[d["temp_imputada"] == 0].copy()
medidas["tasa"] = medidas["defectuosas"] / medidas["producidas"] * 100
for m in ["M-03", "M-01"]:
    s = medidas[medidas["id_maquina"] == m]
    print(f"Correlación temperatura–tasa de defectos en {m} (n={len(s)}): {s['temperatura_c'].corr(s['tasa']):.2f}")

,registros,temp_media_c,producidas,defectuosas,tasa_pct
turno,,,,,
Tarde,23,38.5,26923,1736,6.45
Mañana,22,34.5,26683,925,3.47
Noche,22,30.7,27402,814,2.97


SQL y pandas coinciden.

Correlación temperatura–tasa de defectos en M-03 (n=67): 0.81
Correlación temperatura–tasa de defectos en M-01 (n=64): -0.04


**Hallazgo.** En la M-03 los defectos se concentran en la **tarde**: 6,45 %, casi el doble que en la mañana (3,47 %) y más del doble que en la noche (2,97 %). Es también el turno más caliente (38,5 °C de media, por encima del límite de 38,2 °C del IQR). La correlación temperatura–tasa es **0,81 en la M-03** y **−0,04 en la M-01**. Como la M-03 es la más antigua (2012), una hipótesis es desgaste o refrigeración insuficiente, **pero es solo una hipótesis**: correlación no es causalidad y habría que contrastarla con el historial de mantenimiento (entidad `MANTENIMIENTO` del ERD, sin datos en este dataset).

**Recomendación:** inspeccionar primero la **M-03**, con foco en su comportamiento térmico en el turno de la tarde.

In [15]:
con.close()

## Referencias
[1] IBM, "¿Qué es el mantenimiento predictivo?," IBM Think, 2026. https://www.ibm.com/think/topics/predictive-maintenance  
[2] J. A. González Bonilla, "Taller en clase: Calidad de datos, diagnosticar, limpiar y medir," CORHUILA, Semana 9, Corte 2, 2026.  
[5] W. McKinney, "Data structures for statistical computing in Python," in *Proc. 9th Python in Science Conf. (SciPy)*, 2010, pp. 56–61.  
[6] C. Batini and M. Scannapieco, *Data and Information Quality*. Cham, Switzerland: Springer, 2016.  
(La lista completa está en el `README.md`.)